# Lab Day 10 — Qwen3-VL road damage grounding (Kaggle)

Chạy trên Kaggle Notebook với GPU T4. Bật Internet và GPU trong Notebook settings. Dataset/API credentials chỉ đặt trong Kaggle Secrets hoặc biến môi trường; không ghi khóa vào notebook. Workflow ở `kaggle_lab.py`; giữ nguyên `scripts/` và `tests/` của đề.


## 0. Cài môi trường và lấy mã nguồn

Mở Kaggle Notebook trong repository hoặc upload notebook cùng `kaggle_lab.py`, thư mục `scripts/`, `configs/` và file requirements. Kaggle thường đã cài PyTorch CUDA; cài các package còn lại theo phiên bản đề ghim.


In [ ]:
!nvidia-smi
!pip -q install -r requirements-colab.txt
import os, sys, json, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'Kaggle Notebook settings → Accelerator → GPU'


## 1. Tải dữ liệu từ Kaggle

Thêm Kaggle API credentials vào Kaggle Secrets (`KAGGLE_USERNAME`, `KAGGLE_KEY`) và expose chúng vào notebook, hoặc attach dataset trực tiếp rồi dùng `--source`.


In [ ]:
# Kaggle Secrets API; credentials không được in hoặc lưu vào file
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = secrets.get_secret('KAGGLE_KEY')
!python scripts/prepare_data.py --kaggle
meta = json.load(open('data/splits/meta.json', encoding='utf-8'))
print('dataset_version:', meta['dataset_version'])
print(json.dumps(meta['counts'], indent=2))


## 2. Nhãn và kiểm tra dữ liệu

Hàm YOLO → JSON 0–1000, thứ tự box ổn định và vẽ sáu ví dụ đã được triển khai trong `kaggle_lab.py`. Đây là bước kiểm tra dữ liệu trước huấn luyện.


In [ ]:
from kaggle_lab import *
show_split_examples(6)
train_rows = read_jsonl('data/splits/train.jsonl')
print('sample answer:', target_json(train_rows[0]))


## 3. E0 — zero-shot

Tải model 4-bit NF4; `visual` và `lm_head` được giữ fp16. Lần đầu tải model có thể mất thời gian.


In [ ]:
model, processor = load_model()
e0_pred = run_inference(model, processor, split='test', out='work/e0.jsonl')
score(e0_pred, 'E0')
del model, processor; torch.cuda.empty_cache()


## 4. E1, E2, E3 — huấn luyện QLoRA

Mỗi lần chạy cấu hình một thí nghiệm; mặc định 300 optimizer steps, batch 1 và tích lũy gradient 4 lần. E2 mở merger/deepstack merger ở fp32; E3 thêm LoRA ViT. Adapter và projector được lưu dưới `work/` để tránh commit trọng số. Sau mỗi thí nghiệm, copy thư mục adapter sang Kaggle Dataset hoặc persistent output nếu cần giữ qua phiên.


In [ ]:
# Chạy mỗi experiment trong phiên riêng nếu cần giải phóng VRAM. Đổi 'e1' thành 'e2' hoặc 'e3'.
adapter_dir, run_meta = train(experiment='e1', rank=16, max_steps=300)


In [ ]:
# Lặp lại riêng cho E2 và E3; ghi lại adapter_dir và run_meta của từng lần.
# adapter_e2, meta_e2 = train(experiment='e2', rank=16, max_steps=300)
# adapter_e3, meta_e3 = train(experiment='e3', rank=16, max_steps=300)


## 5. Suy luận và chấm E1–E3

Nạp lại model gốc, adapter và projector đã lưu; chạy toàn bộ test đúng một lần cho mỗi cấu hình. Evidence được sinh từ dữ liệu và dự đoán thật ở phiên Kaggle của bạn.


In [ ]:
# Dùng adapter_dir còn trong phiên; nếu khởi động lại, trỏ tới đường dẫn đã khôi phục từ Kaggle Dataset.
model, processor = load_model()
pred_path = run_inference(model, processor, 'test', adapter=str(adapter_dir), out='work/e1.jsonl')
score(pred_path, 'E1', run_meta)
del model, processor; torch.cuda.empty_cache()


In [ ]:
# Thực hiện tương tự cho E2/E3 (nạp adapter_e2/meta_e2 hoặc adapter_e3/meta_e3).
# model, processor = load_model()
# p2 = run_inference(model, processor, 'test', adapter=str(adapter_e2), out='work/e2.jsonl')
# score(p2, 'E2', meta_e2)  # ghi report/evidence/e2.json
# del model, processor; torch.cuda.empty_cache()
# model, processor = load_model()
# p3 = run_inference(model, processor, 'test', adapter=str(adapter_e3), out='work/e3.jsonl')
# score(p3, 'E3', meta_e3)  # ghi report/evidence/e3.json
# del model, processor; torch.cuda.empty_cache()


## 6. E4 — ablation rank

Thay đổi duy nhất rank so với E1: chạy rank 4 và rank 64, cùng seed, dữ liệu, số bước, prompt và cách chấm. Cả hai runs cần được huấn luyện và chấm trên test để tạo hai file evidence.


In [ ]:
# Ví dụ chạy mỗi rank ở phiên riêng. Đổi adapter path theo thư mục đã lưu.
import subprocess
# adapter_r4, meta_r4 = train('e1', rank=4, max_steps=300)
# adapter_r64, meta_r64 = train('e1', rank=64, max_steps=300)
# model, processor = load_model(); pr4 = run_inference(model, processor, 'test', str(adapter_r4), 'work/e4_r4.jsonl')
# subprocess.run([sys.executable, 'scripts/score.py', '--pred', str(pr4), '--name', 'E4-rank4', '--out', 'report/evidence/e4_rank_4.json', '--meta', f"trainable_params={meta_r4['trainable_params']}", '--meta', f"peak_vram_gb={meta_r4['peak_vram_gb']}", '--meta', f"train_minutes={meta_r4['train_minutes']}"], check=True)
# del model, processor; torch.cuda.empty_cache()
# model, processor = load_model(); pr64 = run_inference(model, processor, 'test', str(adapter_r64), 'work/e4_r64.jsonl')
# subprocess.run([sys.executable, 'scripts/score.py', '--pred', str(pr64), '--name', 'E4-rank64', '--out', 'report/evidence/e4_rank_64.json', '--meta', f"trainable_params={meta_r64['trainable_params']}", '--meta', f"peak_vram_gb={meta_r64['peak_vram_gb']}", '--meta', f"train_minutes={meta_r64['train_minutes']}"], check=True)


## 7. Bằng chứng và báo cáo

Tải về các evidence JSON, adapter metadata (`run.json`) và ít nhất sáu ảnh minh họa dự đoán có vẽ box. Chỉ điền `REPORT.md` sau khi có các kết quả thực đo; không dùng các số ước lượng trong hướng dẫn như kết quả. Chạy `python scripts/check_submission.py` trên máy/repo có split data để kiểm tra cấu trúc trước khi nộp.
